In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def split_time(obj):
    times_s = []
    for time in obj:
        hms = time.split(":")
        s = int(hms[0])*60+int(hms[1])
        times_s.append(s)
    return times_s

In [ ]:
raw_od =  pd.read_excel("../Data/Terminator_Strength-250305.xlsx", sheet_name="OD600")
raw_gfp =  pd.read_excel("../Data/Terminator_Strength-250305.xlsx", sheet_name="GFP")
raw_rfp =  pd.read_excel("../Data/Terminator_Strength-250305.xlsx", sheet_name="RFP")

raw_od["Time"] = split_time(raw_od["Time"])
raw_gfp["Time"] = split_time(raw_gfp["Time"])
raw_rfp["Time"] = split_time(raw_rfp["Time"])

In [ ]:
def combine(list1, list2, sep=""):
    new = []
    for el in list1:
        for el2 in list2:
            new.append(el+sep+el2)
    return(new)

In [ ]:
letters = ["A", "B", "C", "D", "E", "F", "G", "H"]
numbers = [str(num+1) for num in range(12)]
all_wells = combine(letters, numbers)
sample_wells = []
for well in all_wells:
    if not well[1:] in ["10", "11", "12"]:
        sample_wells.append(well)

blank_wells = ["G11", "G12", "H11", "H12"]
af_wells = ["A10", "A11", "A12"]
samples = combine(["0", "1", "2.5", "5", "10", "25", "50", "100"], ["TT7hyb1", "noTer", "RPU"], sep="_")

In [ ]:
def raw_stats(df):
    mean_df = pd.DataFrame()
    mean_df.index = df.index
    mean_df["Blank"] = df[blank_wells].mean(axis=1)
    mean_df["AF"] = df[af_wells].mean(axis=1)
    for idx, el in enumerate(samples):
        mean_df[el] = df[sample_wells[idx*3:idx*3+3]].mean(axis=1)
    
    
    std_df = pd.DataFrame()
    std_df.index = df.index
    std_df["Blank"] = df[["G11", "G12", "H11", "H12"]].std(axis=1)
    std_df["AF"] = df[["A10", "A11", "A12"]].std(axis=1)
    for idx, el in enumerate(samples):
        std_df[el] = df[sample_wells[idx*3:idx*3+3]].std(axis=1)
        #print(el, "->", sample_wells[idx*3:idx*3+3])

    return mean_df, std_df

In [ ]:
def blank(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df["AF"] = df["AF"] - df["Blank"]
        blanked_df[samples] = df[samples].sub(df["Blank"], axis=0)
    else:
        blanked_df["AF"] = np.sqrt(df["AF"]**2 + df["Blank"]**2)
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["Blank"], 2), axis=0))

    return blanked_df

In [ ]:
def remove_af(df, std = False):
    blanked_df = pd.DataFrame()
    blanked_df.index = df.index
    if not std:
        blanked_df[samples] = df[samples].sub(df["AF"], axis=0)
    else:
        blanked_df[samples] = np.sqrt(np.power(df[samples], 2).add(np.power(df["AF"], 2), axis=0))

    return blanked_df

In [ ]:
fig, axs = plt.subplots(8, 4, figsize = (10, 10), constrained_layout=True)
fig.suptitle("RAW_OD600")
fig.text(0.5, 0.00, 'Time (min)', ha='center')
fig.text(0.00, 0.5, 'OD600', va='center', rotation='vertical')
df = raw_od
for i in range(int(96/3)):
    axs[i//4, i%4].plot(df.index, df[all_wells[i*3:i*3+3]], color='grey')
    axs[i//4, i%4].set_ylim(10**-1, 10**0)
    axs[i//4, i%4].set_xlim(0, df.index.max())
    axs[i//4, i%4].set_yscale('log')
plt.show()

In [ ]:
fig, axs = plt.subplots(8, 4, figsize = (10, 10), constrained_layout=True)
fig.suptitle("RAW_sfGFP")
df = raw_gfp
for i in range(int(96/3)):
    axs[i//4, i%4].plot(df.index, df[all_wells[i*3:i*3+3]], color='g')
    axs[i//4, i%4].set_ylim(10**4, 10**7)
    axs[i//4, i%4].set_xlim(0, df.index.max())
    axs[i//4, i%4].set_yscale('log')
plt.show()

In [ ]:
fig, axs = plt.subplots(8, 4, figsize = (10, 10), constrained_layout=True)
fig.suptitle("RAW_mScarlet")
df = raw_rfp
for i in range(int(96/3)):
    axs[i//4, i%4].plot(df.index, df[all_wells[i*3:i*3+3]], color='r')
    axs[i//4, i%4].set_ylim(10**3, 10**7)
    axs[i//4, i%4].set_xlim(0, df.index.max())
    axs[i//4, i%4].set_yscale('log')
plt.show()

In [ ]:
raw_od[["A2", "E7", "A10"]] = np.nan
raw_od = raw_od.set_index('Time')
raw_gfp[["A2", "E7", "A10"]] = np.nan
raw_gfp = raw_gfp.set_index('Time')
raw_rfp[["A2", "E7", "A10"]] = np.nan
raw_rfp = raw_rfp.set_index('Time')

In [ ]:
mean_od, std_od = raw_stats(raw_od)
mean_gfp, std_gfp = raw_stats(raw_gfp)
mean_rfp, std_rfp = raw_stats(raw_rfp)

In [ ]:
blanked_mean_od = blank(mean_od)
blanked_std_od = blank(std_od, std=True)

blanked_mean_gfp = blank(mean_gfp)
blanked_std_gfp = blank(std_gfp, std=True)

blanked_mean_rfp = blank(mean_rfp)
blanked_std_rfp = blank(std_rfp, std=True)

In [ ]:
plt.figure()
plt.title("sanity check")
plt.errorbar(blanked_mean_gfp.index, blanked_mean_gfp["AF"], yerr=blanked_std_gfp["AF"])
plt.errorbar(blanked_mean_gfp.index, blanked_mean_gfp["0_TT7hyb1"], yerr=blanked_std_gfp["0_TT7hyb1"])
plt.show()

In [ ]:
blanked_mean_gfp.index = blanked_mean_od.index
blanked_mean_gfp.index = blanked_mean_od.index

blanked_std_gfp.index = blanked_std_od.index
blanked_std_rfp.index = blanked_std_od.index

In [ ]:
mean_G_OD = blanked_mean_gfp/blanked_mean_od
std_G_OD = abs(mean_G_OD) * np.sqrt((blanked_std_gfp/blanked_mean_gfp)**2 + (blanked_std_od/blanked_mean_od)**2)

mean_R_OD = blanked_mean_rfp/blanked_mean_od
std_R_OD = abs(mean_R_OD) * np.sqrt((blanked_std_rfp/blanked_mean_rfp)**2 + (blanked_std_od/blanked_mean_od)**2)

In [ ]:
mean_G_OD_AF = remove_af(mean_G_OD)
std_G_OD_AF = remove_af(std_G_OD, std=True)

mean_R_OD_AF = remove_af(mean_R_OD)
std_R_OD_AF = remove_af(std_R_OD, std=True)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3), constrained_layout=True)
fig.suptitle("Autofluorescence/OD600")

axs[0].errorbar(mean_G_OD.index, mean_G_OD["AF"], yerr=std_G_OD["AF"], color='g')
axs[0].set_ylim(0, 10**6)
axs[0].set_xlim(0, max(mean_G_OD.index))
axs[0].set_title("sfGFP")

axs[1].errorbar(mean_R_OD.index, mean_R_OD["AF"], yerr=std_R_OD["AF"], color='r')
axs[1].set_ylim(0, 10**6)
axs[1].set_xlim(0, max(mean_R_OD.index))
axs[1].set_title("mScarlet")

plt.show()

In [ ]:
fig, axs = plt.subplots(8, 3, figsize=(15, 15), constrained_layout=True)
fig.suptitle("sfGFP/OD")
dfs = [mean_G_OD_AF, std_G_OD_AF]
for i in range(24):
    axs[i//3, i%3].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]], fmt='g')
    axs[i//3, i%3].set_yscale("log")
    axs[i//3, i%3].set_ylim(10**4, 10**8)
    axs[i//3, i%3].set_xlim(0, max(dfs[0].index))
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 2), constrained_layout=True)
#fig.suptitle("sfGFP/OD")
dfs = [mean_G_OD_AF, std_G_OD_AF]
for i in range(24):
    match samples[i].split("_")[1]:
        case "TT7hyb1":
            col = 0
        case "noTer":
            col = 1
        case "RPU":
            col = 2
    axs[col].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]])
    axs[col].set_yscale("log")
    axs[col].set_ylim(10**4, 10**8)
    axs[col].set_xlim(0, max(dfs[0].index))

axs[0].set_title("T7hyb1")
axs[0].set_ylabel("sfGFP/OD600 (a.u.)")
axs[1].set_title("20bp")
axs[2].set_title("RPU")

for ax in axs:
    ax.set_axisbelow(True)
    ax.grid()

plt.savefig("firstest_GFP.png")
plt.show()

In [ ]:
fig, axs = plt.subplots(8, 3, figsize=(15, 15), constrained_layout=True)
fig.suptitle("mScarlet/OD")
dfs = [mean_R_OD_AF, std_R_OD_AF]
for i in range(24):
    axs[i//3, i%3].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]], fmt='r')
    axs[i//3, i%3].set_yscale("log")
    axs[i//3, i%3].set_ylim(10**3, 10**8)
    axs[i//3, i%3].set_xlim(0, max(dfs[0].index))
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 2), constrained_layout=True)
dfs = [mean_R_OD_AF, std_R_OD_AF]
for i in range(24):
    match samples[i].split("_")[1]:
        case "TT7hyb1":
            col = 0
        case "noTer":
            col = 1
        case "RPU":
            col = 2
    axs[col].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]])
    axs[col].set_yscale("log")
    axs[col].set_ylim(10**3, 10**8)
    axs[col].set_xlim(0, max(dfs[0].index))
    
axs[0].set_ylabel("mScarlet/OD600 (a.u.)")

for ax in axs:
    ax.set_axisbelow(True)
    ax.grid()

fig.text(0.33, -.05, "Time (min)")
plt.savefig("firsttest_RFP.png", bbox_inches='tight')
plt.show()

In [ ]:
mean_RG = mean_R_OD_AF/mean_G_OD_AF
std_RG = abs(mean_RG) * np.sqrt((std_G_OD_AF/mean_G_OD_AF)**2 + (std_R_OD_AF/mean_R_OD_AF)**2)

In [ ]:
fig, axs = plt.subplots(8, 3, figsize=(15, 15), constrained_layout=True)
fig.suptitle("RFP/GFP")
dfs = [mean_RG, std_RG]

for i in range(24):
    axs[i//3, i%3].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]])
    axs[i//3, i%3].set_yscale("log")
    axs[i//3, i%3].set_ylim(10**-2, 10**1)
    axs[i//3, i%3].set_xlim(0, max(dfs[0].index))
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3), constrained_layout=True)
fig.suptitle("RFP/GFP")
fig.text(0.5, -0.1, 'Time (min)', ha='center')
dfs = [mean_RG, std_RG]
for i in range(24):
    match samples[i].split("_")[1]:
        case "TT7hyb1":
            col = 0
        case "noTer":
            col = 1
        case "RPU":
            col = 2
    axs[col].errorbar(dfs[0].index, dfs[0][samples[i]], yerr=dfs[1][samples[i]])
    axs[col].set_yscale("log")
    axs[col].set_ylim(10**-2, 10**1)
    axs[col].set_xlim(0, max(dfs[0].index))
    
axs[0].set_title("T7hyb1")
axs[1].set_title("noTer")
axs[2].set_title("RPU")

axs[2].legend(["0uM Cuma", "1uM Cuma", "2.5uM Cuma", "5uM Cuma", "10uM Cuma", "25uM Cuma", "50uM Cuma", "100uM Cuma"])

plt.savefig("firsttest_cuma.png")
plt.show()

In [ ]:
t_345 = {"TT7hyb1":{"meanGFP":[], "meanRFP":[], "stdGFP":[], "stdRFP":[], "meanRG":[], "stdRG":[]},
         "noTer":{"meanGFP":[], "meanRFP":[], "stdGFP":[], "stdRFP":[], "meanRG":[], "stdRG":[]},
         "RPU":{"meanGFP":[], "meanRFP":[], "stdGFP":[], "stdRFP":[], "meanRG":[], "stdRG":[]}}
for sample in samples:
    conc, name = sample.split("_")
    t_345[name]["meanGFP"].append(mean_G_OD_AF.loc[345, sample])
    t_345[name]["meanRFP"].append(mean_R_OD_AF.loc[345, sample])
    t_345[name]["meanRG"].append(mean_RG.loc[345, sample])
    t_345[name]["stdGFP"].append(std_G_OD_AF.loc[345, sample])
    t_345[name]["stdRFP"].append(std_R_OD_AF.loc[345, sample])
    t_345[name]["stdRG"].append(std_RG.loc[345, sample])
    

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 3), constrained_layout=True)

for key in t_345.keys():
    ax[0].errorbar(t_345[key]["meanGFP"],
                t_345[key]["meanRFP"],
                xerr = t_345[key]["stdGFP"],
                yerr = t_345[key]["stdRFP"],
                marker="o", ecolor="k")

    ax[1].errorbar([0, 1, 2.5, 5, 10, 25, 50, 100],
                   t_345[key]["meanRG"],
                   yerr=t_345[key]["stdRG"],
                   marker="o", ecolor="k")
    
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlim(10**4, 10**8)
ax[0].set_ylim(10**3, 10**8)
ax[0].set_xlabel("sfGFP / OD600 (a.u.)")
ax[0].set_ylabel("mScarlet / OD600 (a.u.)")
ax[0].legend(["T7hyb1", "20bp", "RPU"])
ax[0].grid()

ax[1].set_xscale("symlog")
ax[1].set_yscale("log")
ax[1].set_xlabel("[Cuminic Acid] (uM)")
ax[1].set_ylabel("mScarlet/sfGFP")
ax[1].grid()

plt.savefig("firsttest_RG.png")
plt.show()

In [ ]:
RPU_samples = combine(["0", "1", "2.5", "5", "10", "25", "50", "100"], ["RPU"], sep="_")

mean_R_au = mean_R_OD_AF.drop(RPU_samples, axis=1)
std_R_au = std_R_OD_AF.drop(RPU_samples, axis=1)


mean_G_RPU = pd.DataFrame()
mean_G_RPU.index = mean_G_OD_AF.index

std_G_RPU = pd.DataFrame()
std_G_RPU.index = mean_G_OD_AF.index

for sample in samples:
    conc, name = sample.split("_")
    mean_G_RPU[sample] = mean_G_OD_AF[sample] / mean_G_OD_AF[conc+"_RPU"]
    std_G_RPU[sample] = mean_G_RPU[sample] * np.sqrt((std_G_OD_AF[sample]/mean_G_OD_AF[sample])**2+(std_G_OD_AF[conc+"_RPU"]/mean_G_OD_AF[conc+"_RPU"])**2)

mean_G_RPU = mean_G_RPU.drop(RPU_samples, axis=1)
std_G_RPU = std_G_RPU.drop(RPU_samples, axis=1)

In [ ]:
t_345_RPU = {"TT7hyb1":{"meanGFP":[], "meanRFP":[], "stdGFP":[], "stdRFP":[], "meanRG":[], "stdRG":[]},
         "noTer":{"meanGFP":[], "meanRFP":[], "stdGFP":[], "stdRFP":[], "meanRG":[], "stdRG":[]}}
no_RPU_samples = combine(["0", "1", "2.5", "5", "10", "25", "50", "100"], ["TT7hyb1", "noTer"], sep="_")
for sample in no_RPU_samples:
    conc, name = sample.split("_")
    t_345_RPU[name]["meanGFP"].append(mean_G_RPU.loc[345, sample])
    t_345_RPU[name]["meanRFP"].append(mean_R_au.loc[345, sample])
    #t_345_RPU[name]["meanRG"].append(mean_RG.loc[345, sample])
    t_345_RPU[name]["stdGFP"].append(std_G_RPU.loc[345, sample])
    t_345_RPU[name]["stdRFP"].append(std_R_au.loc[345, sample])
    #t_345_RPU[name]["stdRG"].append(std_RG.loc[345, sample])
    

In [ ]:
fig, ax = plt.subplots()

for key in t_345_RPU.keys():
    ax.errorbar(t_345_RPU[key]["meanGFP"],
                t_345_RPU[key]["meanRFP"],
                xerr = t_345_RPU[key]["stdGFP"],
                yerr = t_345_RPU[key]["stdRFP"],
                marker="o", ecolor="k")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_ylim(10**5, 3*10**7)
ax.set_xlabel("sfGFP (RPU)")
ax.set_ylabel("mScarlet / OD600 (a.u.)")
ax.legend(t_345.keys())
ax.grid()

plt.show()

In [ ]:
t_345_RPU["TT7hyb1"]["RFP_growth"] = []
t_345_RPU["noTer"]["RFP_growth"] = []

t_345_RPU["TT7hyb1"]["RFP_growth_std"] = []
t_345_RPU["noTer"]["RFP_growth_std"] = []

for idx, point in enumerate(t_345_RPU["TT7hyb1"]["meanRFP"]):
    t_345_RPU["TT7hyb1"]["RFP_growth"].append(point - t_345_RPU["TT7hyb1"]["meanRFP"][0])
for idx, point in enumerate(t_345_RPU["noTer"]["meanRFP"]):
    t_345_RPU["noTer"]["RFP_growth"].append(point - t_345_RPU["noTer"]["meanRFP"][0])

for idx, std in enumerate(t_345_RPU["TT7hyb1"]["stdRFP"]):
    t_345_RPU["TT7hyb1"]["RFP_growth_std"].append(np.sqrt(std**2 + t_345_RPU["TT7hyb1"]["stdRFP"][0]**2))
for idx, std in enumerate(t_345_RPU["noTer"]["stdRFP"]):
    t_345_RPU["noTer"]["RFP_growth_std"].append(np.sqrt(std**2 + t_345_RPU["noTer"]["stdRFP"][0]**2))

t_345_RPU["TT7hyb1"]["RFP_growth"] = t_345_RPU["TT7hyb1"]["RFP_growth"][3:]
t_345_RPU["noTer"]["RFP_growth"] = t_345_RPU["noTer"]["RFP_growth"][3:]

t_345_RPU["TT7hyb1"]["RFP_growth_std"] = t_345_RPU["TT7hyb1"]["RFP_growth_std"][3:]
t_345_RPU["noTer"]["RFP_growth_std"] = t_345_RPU["noTer"]["RFP_growth_std"][3:]

In [ ]:
t_345_RPU["TT7hyb1"]["meanRG"] = []
t_345_RPU["noTer"]["meanRG"] = []

t_345_RPU["TT7hyb1"]["stdRG"] = []
t_345_RPU["noTer"]["stdRG"] = []

for idx, point in enumerate(t_345_RPU["TT7hyb1"]["RFP_growth"]):
    t_345_RPU["TT7hyb1"]["meanRG"].append(point / t_345_RPU["TT7hyb1"]["meanGFP"][0])
for idx, point in enumerate(t_345_RPU["noTer"]["RFP_growth"]):
    t_345_RPU["noTer"]["meanRG"].append(point / t_345_RPU["noTer"]["meanGFP"][0])

#t_345_RPU["TT7hyb1"]["meanRG"][0] = 1.0
#t_345_RPU["noTer"]["meanRG"][0] = 1.0

for idx, std in enumerate(t_345_RPU["TT7hyb1"]["RFP_growth_std"]):
    t_345_RPU["TT7hyb1"]["stdRG"].append(t_345_RPU["TT7hyb1"]["meanRG"][idx]*np.sqrt((std/t_345_RPU["TT7hyb1"]["RFP_growth"][idx])**2 + (t_345_RPU["TT7hyb1"]["stdGFP"][idx]/t_345_RPU["TT7hyb1"]["meanGFP"][idx])**2))
for idx, std in enumerate(t_345_RPU["noTer"]["RFP_growth_std"]):
    t_345_RPU["noTer"]["stdRG"].append(t_345_RPU["noTer"]["meanRG"][idx]*np.sqrt((std/t_345_RPU["noTer"]["RFP_growth"][idx])**2 + (t_345_RPU["noTer"]["stdGFP"][idx]/t_345_RPU["noTer"]["meanGFP"][idx])**2))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)

for key in t_345_RPU.keys():
    ax[0].errorbar(t_345_RPU[key]["meanGFP"][3:],
                t_345_RPU[key]["RFP_growth"],
                xerr = t_345_RPU[key]["stdGFP"][3:],
                yerr = t_345_RPU[key]["RFP_growth_std"],
                marker="o", ecolor="k")

    ax[1].errorbar([0, 1, 2.5, 5, 10, 25, 50, 100][3:],
                   t_345_RPU[key]["meanRG"],
                   yerr=t_345_RPU[key]["stdRG"],
                   marker="o", ecolor="k")
  
ax[0].set_xscale("log")
ax[0].set_yscale("symlog")
#ax[0].set_xlim(10**4, 10**8)
ax[0].set_ylim(10**4, 10**8)
ax[0].set_xlabel("sfGFP / OD600 (RPU)")
ax[0].set_ylabel("mScarlet / OD600 (a.u.)")
ax[0].legend(t_345_RPU.keys())
ax[0].grid()

ax[1].set_xscale("symlog")
ax[1].set_yscale("symlog")
ax[1].set_xlabel("[Cuminic Acid] (uM)")
ax[1].set_ylabel("mScarlet/sfGFP")
ax[1].set_xlim(10**0, 2 * 10**2)
ax[1].grid()

plt.show()

In [ ]:
meanTe = []
stdTe = []
for idx, point in enumerate(t_345_RPU["TT7hyb1"]["meanRG"]):
    print("1 - (", point, "/", t_345_RPU["noTer"]["meanRG"][idx], ") =")
    mean_calc = point / t_345_RPU["noTer"]["meanRG"][idx]
    meanTe.append(100*(1 - mean_calc))
    print(meanTe[idx])
    std = t_345_RPU["TT7hyb1"]["stdRG"][idx]
    std_calc = mean_calc*np.sqrt((std/t_345_RPU["TT7hyb1"]["meanRG"][idx])**2 + (t_345_RPU["noTer"]["stdRG"][idx]/t_345_RPU["noTer"]["meanRG"][idx])**2)
    stdTe.append(std_calc *100)
    print(std_calc)

In [ ]:
fig, ax = plt.subplots()
fig.suptitle("T7hyb1 Termination Efficiency")
ax.errorbar([0, 1, 2.5, 5, 10, 25, 50, 100][3:],
            meanTe,
            yerr = stdTe,
            color="k", marker="o", ecolor="k")
ax.set_xscale("log")
ax.set_yscale("linear")
ax.set_ylabel("Termination Efficiency")
ax.set_xlabel("[Cuminic Acid] (uM)")
ax.set_yticks(np.linspace(0,100,11))
ax.set_ylim(0, 100)
ax.set_xlim(10**0, 2 * 10**2)
ax.grid()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3),constrained_layout=True)
#fig.suptitle("T7hyb1 Termination Efficiency")
ax.errorbar([0.006933842, 0.020000001, 0.065229006, 0.516514, 7.543343, 15.161667, 17.154509, 18.54178][3:],
            meanTe,
            yerr = stdTe,
            color="k", marker="o", ecolor="k")
ax.set_xscale("log")
ax.set_ylabel("Termination Efficiency (%)")
ax.set_xlabel("Input (RPU)")
ax.set_yticks(np.linspace(0,100,11))
ax.set_ylim(0, 100)
ax.set_axisbelow(True)
ax.grid()
plt.savefig("firsttest_Te.png")
plt.show()

In [ ]:
stdTe

In [ ]:
fig, ax = plt.subplots()
fig.suptitle("T7hyb1 Termination Efficiency")
ax.errorbar(t_345_RPU[key]["meanGFP"][3:],
            meanTe,
            xerr = t_345_RPU[key]["stdGFP"][3:],
            yerr = stdTe,
            color="k", marker="o", ecolor="k")
ax.set_ylabel("Termination Efficiency")
ax.set_xlabel("sfGFP/OD600 (RPU)")
ax.set_yticks(np.linspace(0,1,11))
ax.set_ylim(0, 1)
ax.grid()
plt.show()

In [ ]:
samples = ["0_noTer", "5_noTer", "100_noTer", "0_TT7hyb1", "5_TT7hyb1", "100_TT7hyb1"]
t = blanked_mean_od.index[23]
fig, ax = plt.subplots()
fig.suptitle("OD600 at t = "+str(t)+" (min)")
ax.bar(samples,
       [blanked_mean_od.loc[t, col] for col in samples],
       color="grey")
ax.set_xticklabels(samples, rotation=45, ha="right", rotation_mode="anchor")

ax.errorbar(samples,
            [blanked_mean_od.loc[t, col] for col in samples],
            yerr = [blanked_std_od.loc[t, col] for col in samples],
            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
plt.show()

In [ ]:
deriv_bl_m_od = pd.DataFrame(index = blanked_mean_od.index[1:])
deriv_bl_s_od = pd.DataFrame(index = blanked_std_od.index[1:])
for col in blanked_mean_od.columns:
    col_list = blanked_mean_od[col].to_list()
    deriv_list = []
    prev_num = 0
    for num in col_list:
        if prev_num != 0:
            deriv_list.append(num-prev_num)
        prev_num = num

    col_list = blanked_std_od[col].to_list()
    std_list = []
    prev_num = 0
    for num in col_list:
        if prev_num != 0:
            std_list.append(np.sqrt(num**2+prev_num**2))
        prev_num = num
    deriv_bl_m_od[col] = deriv_list
    deriv_bl_s_od[col] = std_list

In [ ]:
samples = ["0_noTer", "5_noTer", "100_noTer", "0_TT7hyb1", "5_TT7hyb1", "100_TT7hyb1"]
fig, ax = plt.subplots()
fig.suptitle("max OD600 growth rate (OD600/15min)")
ax.bar(samples,
       [max(deriv_bl_m_od[col]) for col in samples],
       color="grey")
ax.set_xticklabels(samples, rotation=45, ha="right", rotation_mode="anchor")

ax.errorbar(samples,
            [max(deriv_bl_m_od[col]) for col in samples],
            yerr = [deriv_bl_s_od.iloc[deriv_bl_m_od[col].to_list().index(max(deriv_bl_m_od[col]))][col] for col in samples],
            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
plt.show()

In [ ]:
samples = ["0_noTer", "5_noTer", "100_noTer", "0_TT7hyb1", "5_TT7hyb1", "100_TT7hyb1"]
fig, ax = plt.subplots()
fig.suptitle("time to max growth rate")
ax.bar(samples,
       [deriv_bl_m_od.index[deriv_bl_m_od[col].to_list().index(max(deriv_bl_m_od[col]))] for col in samples],
       color="grey")
ax.set_xticklabels(samples, rotation=45, ha="right", rotation_mode="anchor")
ax.set_ylim(300, 400)

plt.show()

In [ ]:
samples = combine()
t = blanked_mean_od.index[23]
fig, ax = plt.subplots()
fig.suptitle("OD600 at t = "+str(t)+" (min)")
ax.bar(samples,
       [blanked_mean_od.loc[t, col] for col in samples],
       color="grey")
ax.set_xticklabels(samples, rotation=45, ha="right", rotation_mode="anchor")

ax.errorbar(samples,
            [blanked_mean_od.loc[t, col] for col in samples],
            yerr = [blanked_std_od.loc[t, col] for col in samples],
            marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
plt.show()

In [ ]:
deriv_bl_m_od.index[deriv_bl_m_od[col].to_list().index(max(deriv_bl_m_od[col]))]

---

## ENDPOINTS

In [ ]:
raw_od =  pd.read_excel("Terminator_Strength-250328.xlsx", sheet_name="OD600")
raw_gfp =  pd.read_excel("Terminator_Strength-250328.xlsx", sheet_name="GFP")
raw_rfp =  pd.read_excel("Terminator_Strength-250328.xlsx", sheet_name="RFP")

In [ ]:
letters = ["E", "F", "G"]
numbers = [str(num+1) for num in range(12)]
all_wells = combine(letters, numbers)
sample_wells = []
for well in all_wells:
    if not well[1:] in ["10", "11", "12"]:
        sample_wells.append(well)

blank_wells = ["G11", "G12", "H11", "H12"]

samples = ["TT7hyb1_pin", "J23105_pin", "20bp_spacer_pin", "40bp_spacer_pin", "TT7hyb1", "TT7hyb6*", "20bp_spacer", "40bp_spacer", "pSHNJ200"]

In [ ]:
def raw_stats(df):
    mean_df = pd.DataFrame()
    mean_df.index = df.index
    mean_df["Blank"] = df[["G11", "G12", "H11", "H12"]].mean(axis=1)
    mean_df["AF"] = df[["H1", "H2", "H3"]].mean(axis=1)
    for idx, el in enumerate(samples):
        mean_df[el] = df[sample_wells[idx*3:idx*3+3]].mean(axis=1)
    
    
    std_df = pd.DataFrame()
    std_df.index = df.index
    std_df["Blank"] = df[["G11", "G12", "H11", "H12"]].std(axis=1)
    std_df["AF"] = df[["H1", "H2", "H3"]].std(axis=1)
    for idx, el in enumerate(samples):
        std_df[el] = df[sample_wells[idx*3:idx*3+3]].std(axis=1)
        #print(el, "->", sample_wells[idx*3:idx*3+3])

    return mean_df, std_df

In [ ]:
mean_od, std_od = raw_stats(raw_od)
mean_gfp, std_gfp = raw_stats(raw_gfp)
mean_rfp, std_rfp = raw_stats(raw_rfp)

In [ ]:
mean_od

In [ ]:
blanked_mean_od = blank(mean_od)
blanked_std_od = blank(std_od, std=True)

blanked_mean_gfp = blank(mean_gfp)
blanked_std_gfp = blank(std_gfp, std=True)

blanked_mean_rfp = blank(mean_rfp)
blanked_std_rfp = blank(std_rfp, std=True)

In [ ]:
mean_G_OD = blanked_mean_gfp/blanked_mean_od
std_G_OD = abs(mean_G_OD) * np.sqrt((blanked_std_gfp/blanked_mean_gfp)**2 + (blanked_std_od/blanked_mean_od)**2)

mean_R_OD = blanked_mean_rfp/blanked_mean_od
std_R_OD = abs(mean_R_OD) * np.sqrt((blanked_std_rfp/blanked_mean_rfp)**2 + (blanked_std_od/blanked_mean_od)**2)

In [ ]:
mean_R_OD

In [ ]:
mean_G_OD_AF = remove_af(mean_G_OD)
std_G_OD_AF = remove_af(std_G_OD, std=True)

mean_R_OD_AF = remove_af(mean_R_OD)
std_R_OD_AF = remove_af(std_R_OD, std=True)

In [ ]:
mean_G_OD_AF

In [ ]:
mean_G_OD = mean_G_OD.drop("MQ637_pSHNJ200", axis=1)
std_G_OD = std_G_OD.drop("MQ637_pSHNJ200", axis=1)

mean_R_OD = mean_R_OD.drop("MQ637_pSHNJ200", axis=1)
std_R_OD = std_R_OD.drop("MQ637_pSHNJ200", axis=1)

In [ ]:
labels = mean_G_OD.columns
fig, axs = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)

fig.suptitle("Uninduced plasmids in DH10B, without RNaseE site")

axs[0].grid()
axs[0].set_axisbelow(True)
axs[0].bar(labels, mean_G_OD.loc[0], color="g")
axs[0].errorbar(labels, mean_G_OD.loc[0], yerr = std_G_OD.loc[0], marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
axs[0].set_xticks(range(len(labels)))
axs[0].set_xticklabels(labels, rotation=45, ha="right", rotation_mode="anchor")
axs[0].set_yscale('symlog')
axs[0].set_ylim(10**5, 10**8)
axs[0].set_ylabel("sfGFP/OD600 (a.u.)")


axs[1].grid()
axs[1].set_axisbelow(True)
axs[1].bar(labels, mean_R_OD.loc[0], color="r")
axs[1].errorbar(labels, mean_R_OD.loc[0], yerr = std_R_OD.loc[0], marker='none', elinewidth=1, capsize=5, ecolor="k", lw=0)
axs[1].set_xticks(range(len(labels)))
axs[1].set_xticklabels(labels, rotation=45, ha="right", rotation_mode="anchor")
axs[1].set_yscale('symlog')
axs[1].set_ylim(10**2, 10**8)
axs[1].set_ylabel("mScarlet/OD600 (a.u.)")

plt.show()